# S1: PEFT Method Comparison (Full Labels)

**Leaf-JEPA IRP** | Stage 5 — PEFT Adaptation Experiments


**Research Question:** RQ2 — Among LoRA/Adapters/VPT/BitFit, which achieves the best accuracy/parameter tradeoff?

**Experiment flow:**
1. **LR Sweep** — find best LR per method (15 runs, single seed)
2. **Main comparison** — all methods × HP configs × 3 seeds (36 runs)
3. **Computational profiling** — VRAM, time/epoch, inference latency
4. **Summary table + figures**

All methods use **100% of PlantVillage training labels** with the **Leaf-JEPA encoder** (frozen).

## Imports & Configuration

In [1]:
import sys
import json
import time
from pathlib import Path

PROJECT_ROOT = Path("..").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch

from stage5_peft_adaptation_experiments.config_stage5 import *
from stage5_peft_adaptation_experiments.peft_utils import (
    set_seed,
    get_device,
    train_peft,
    build_peft_model,
    count_params,
    print_param_summary,
    profile_inference,
    aggregate_seed_results,
    save_results,
    plot_confusion_matrix,
    plot_pareto,
)

verify_config()
device = get_device()

Stage 5 Configuration Verification
  ✅  PlantVillage root exists
  ✅  PlantDoc root exists
  ✅  Splits directory exists
  ✅  Normalisation stats exist
  ✅  NORM_MEAN matches JSON
  ✅  NORM_STD matches JSON
  ✅  I-JEPA checkpoint exists
  ✅  Leaf-JEPA checkpoint exists
  ✅  Baselines directory exists
  ✅  WANDB_ENTITY set

  10/10 checks passed
  🚀  All checks passed — ready to run Stage 5
Using GPU: NVIDIA H200
VRAM: 150.0 GB


In [2]:
SET1_DIR = PEFT_DIR / "set1"
SET1_DIR.mkdir(parents=True, exist_ok=True)

(FIGURES_DIR / "set1_confusion_matrices").mkdir(parents=True, exist_ok=True)

print(f"Set 1 output directory: {SET1_DIR}")
print(f"Figures directory:      {FIGURES_DIR}")

Set 1 output directory: /workspace/leaf-jepa/stage5_peft_adaptation_experiments/outputs/peft/set1
Figures directory:      /workspace/leaf-jepa/stage5_peft_adaptation_experiments/outputs/figures


## Verify Model Construction

Before running any experiments, verify each PEFT method builds correctly
and produces the expected parameter counts.

In [3]:
# Quick verification: build each method once and check param counts
print("PEFT Method Verification")
print("=" * 60)

test_configs = [
    # ("lora",        {"rank": 8}),
    ("adapter",     {"bottleneck_dim": 64}),
    ("vpt_shallow", {"num_prompts": 50}),
    ("vpt_deep",    {"num_prompts": 50, "num_layers": VIT_DEPTH}),
    ("bitfit",      {}),
    ("linear_probe", {}),
]

for method, kwargs in test_configs:
    model, params = build_peft_model(
        method=method,
        checkpoint_path=LEAF_JEPA_CHECKPOINT,
        model_name=VIT_MODEL_NAME,
        embed_dim=VIT_EMBED_DIM,
        num_classes=NUM_CLASSES,
        device=device,
        **kwargs,
    )
    
    # Verify encoder is frozen (except PEFT params)
    encoder_frozen = sum(
        1 for n, p in model.encoder.named_parameters()
        if p.requires_grad and "lora" not in n and "adapter" not in n
        and "prompt" not in n and "bias" not in n
    )
    if method not in ("full_ft",):
        assert encoder_frozen == 0 or method == "bitfit", \
            f"Encoder has {encoder_frozen} unexpectedly trainable params for {method}!"
    
    del model
    torch.cuda.empty_cache()

print("\n✅ All PEFT methods build correctly with expected param counts.")

PEFT Method Verification

 Building PEFT model: method=adapter
  Encoder loaded from leafjepa-vit-h14-best.pth (frozen)

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
  ADAPTER Classifier
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
  Trainable:     10,784,294
  Frozen:       630,763,520
  Total:        641,547,814
  %% Trainable:      1.6810%%
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

 Building PEFT model: method=vpt_shallow
  Encoder loaded from leafjepa-vit-h14-best.pth (frozen)

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
  VPT_SHALLOW Classifier
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
  Trainable:        112,678
  Frozen:       630,763,520
  Total:        630,876,198
  %% Trainable:      0.0179%%
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

 Building PEFT model: method=vpt_deep
  Encoder loaded from leafjepa-vit-h14-best.pth (frozen)

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
  VPT_DEEP Classifier
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

## Learning Rate Sweep

Find the best LR for each method. Uses seed=42 only (tuning step, not final evaluation).

**15 runs total:** 5 methods × 3 LR candidates

In [ ]:
lr_sweep_results = {}

method_configs = {
    # "lora":        {"rank": 8},
    "adapter":     {"bottleneck_dim": 64},
    "vpt_shallow": {"num_prompts": 50},
    "vpt_deep":    {"num_prompts": 50, "num_layers": VIT_DEPTH},
    "bitfit":      {},
}

for method, kwargs in method_configs.items():
    print(f"\n{'='*60}")
    print(f"  LR SWEEP: {method}")
    print(f"{'='*60}")
    
    lr_sweep_results[method] = {}
    
    for lr in LR_SWEEP_VALUES:
        print(f"\n  Testing LR = {lr}")
        
        result = train_peft(
            method=method,
            checkpoint_path=LEAF_JEPA_CHECKPOINT,
            pv_root=PV_ROOT,
            splits_dir=SPLITS_DIR,
            norm_mean=NORM_MEAN,
            norm_std=NORM_STD,
            model_name=VIT_MODEL_NAME,
            embed_dim=VIT_EMBED_DIM,
            num_classes=NUM_CLASSES,
            fraction=1.0,
            seed=42,
            lr=lr,
            batch_size=BATCH_SIZE,
            max_epochs=MAX_EPOCHS,
            patience=EARLY_STOP_PATIENCE,
            weight_decay=WEIGHT_DECAY,
            warmup_fraction=WARMUP_FRACTION,
            use_amp=USE_AMP,
            gradient_clip=GRADIENT_CLIP,
            num_workers=NUM_WORKERS,
            class_weights_path=CLASS_WEIGHTS_PATH,
            save_dir=SET1_DIR / "lr_sweep",
            run_name=f"LRsweep-{method}-lr{lr}-seed42",
            wandb_project=WANDB_PROJECT,
            wandb_entity=WANDB_ENTITY,
            wandb_group=WANDB_GROUPS["lr_sweep"],
            **kwargs,
        )
        
        lr_sweep_results[method][str(lr)] = result["results"]["val_macro_f1"]

save_results(lr_sweep_results, SET1_DIR / "lr_sweep_results.json")
print("\n✅ LR sweep complete. Results saved.")


  LR SWEEP: adapter

  Testing LR = 0.0001
Using GPU: NVIDIA H200
VRAM: 150.0 GB

  PEFT: adapter | frac=1.0 | seed=42 | lr=0.0001
  Train: 38013 images (fraction=1.0, seed=42)
  Val:   8146 images
  Test:  8146 images

 Building PEFT model: method=adapter
  Encoder loaded from leafjepa-vit-h14-best.pth (frozen)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.



~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
  ADAPTER Classifier
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
  Trainable:     10,784,294
  Frozen:       630,763,520
  Total:        641,547,814
  %% Trainable:      1.6810%%
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


wandb: Currently logged in as: muh-haleef02 (muh-haleef02-inform) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


  Epoch   1/50 | TrLoss: 2.5412 | ValF1: 0.7958 | Acc: 0.8373 | LR: 2.00e-05 | 131.1s


## Select Best LR Per Method

Review sweep results and update `BEST_LR` before continuing.

In [ ]:
print("\nLR Sweep Summary")
print("=" * 60)

selected_lrs = {}

for method, lr_scores in lr_sweep_results.items():
    best_lr = max(lr_scores, key=lr_scores.get)
    best_val_f1 = lr_scores[best_lr]
    selected_lrs[method] = float(best_lr)
    
    print(f"\n  {method}:")
    for lr, f1 in sorted(lr_scores.items()):
        marker = " ◀ BEST" if lr == best_lr else ""
        print(f"    LR = {lr}: val_macro_f1 = {f1:.4f}{marker}")

# Auto-update for this session
for method, lr in selected_lrs.items():
    BEST_LR[method] = lr

save_results(selected_lrs, SET1_DIR / "selected_lrs.json")

print("\n" + "=" * 60)
print("⚠️  Update BEST_LR in config_stage5.py with these values!")
print("    (auto-updated for this session)")
print("=" * 60)

## Define Main Comparison Experiments

Build the experiment grid: all methods × HP variants × 3 seeds.

In [ ]:
experiments = []

# LoRA: ranks 4, 8, 16
for r in LORA_RANKS:
    experiments.append(("lora", f"r{r}", {"rank": r}))

# Adapters: dims 8, 16, 64
for d in ADAPTER_DIMS:
    experiments.append(("adapter", f"d{d}", {"bottleneck_dim": d}))

# VPT-Shallow: lengths 1, 5, 10, 50
for p in VPT_SHALLOW_LENGTHS:
    experiments.append(("vpt_shallow", f"p{p}", {"num_prompts": p}))

# VPT-Deep: 50 tokens
experiments.append(("vpt_deep", f"p{VPT_DEEP_LENGTH}", {
    "num_prompts": VPT_DEEP_LENGTH,
    "num_layers": VIT_DEPTH,
}))

# BitFit: single config
experiments.append(("bitfit", "all", {}))

total_runs = len(experiments) * len(SUBSET_SEEDS)
print(f"Experiment grid: {len(experiments)} configs × {len(SUBSET_SEEDS)} seeds = {total_runs} runs")
print()
for method, tag, kwargs in experiments:
    print(f"  {method:<15} ({tag:<5}) | kwargs = {kwargs}")

## Run Main Comparison (36 runs)

This is the core experiment for RQ2. Each method × HP × seed combination
trains with identical protocol — only the PEFT method differs.

In [ ]:
all_results = []

for exp_idx, (method, hp_tag, kwargs) in enumerate(experiments):
    seed_results = []
    
    for seed in SUBSET_SEEDS:
        run_name = wandb_run_name(method, hp_tag, 1.0, seed)
        
        print(f"\n{'─'*60}")
        print(f"  [{exp_idx * len(SUBSET_SEEDS) + SUBSET_SEEDS.index(seed) + 1}/{total_runs}] {run_name}")
        print(f"{'─'*60}")
        
        result = train_peft(
            method=method,
            checkpoint_path=LEAF_JEPA_CHECKPOINT,
            pv_root=PV_ROOT,
            splits_dir=SPLITS_DIR,
            norm_mean=NORM_MEAN,
            norm_std=NORM_STD,
            model_name=VIT_MODEL_NAME,
            embed_dim=VIT_EMBED_DIM,
            num_classes=NUM_CLASSES,
            fraction=1.0,
            seed=seed,
            lr=BEST_LR[method],
            batch_size=BATCH_SIZE,
            max_epochs=MAX_EPOCHS,
            patience=EARLY_STOP_PATIENCE,
            weight_decay=WEIGHT_DECAY,
            warmup_fraction=WARMUP_FRACTION,
            use_amp=USE_AMP,
            gradient_clip=GRADIENT_CLIP,
            num_workers=NUM_WORKERS,
            class_weights_path=CLASS_WEIGHTS_PATH,
            save_dir=SET1_DIR,
            run_name=run_name,
            wandb_project=WANDB_PROJECT,
            wandb_entity=WANDB_ENTITY,
            wandb_group=WANDB_GROUPS["set1"],
            **kwargs,
        )
        
        result["hp_tag"] = hp_tag
        seed_results.append(result)
        all_results.append(result)
    
    # Aggregate across seeds for this config
    agg = aggregate_seed_results(seed_results)
    print(f"\n  ★ {method} ({hp_tag}): "
          f"Macro-F1 = {agg['macro_f1']['mean']:.4f} ± {agg['macro_f1']['std']:.4f}")

save_results(all_results, PEFT_DIR / "set1_method_comparison.json")
print(f"\n✅ All {total_runs} runs complete. Results saved.")

## Computational Profiling

Profile inference latency for the best HP of each method.

In [ ]:
print("\nInference Profiling (ms/image, batch_size=1)")
print("=" * 60)

profile_results = {}
best_per_method = {}

# Find best HP per method from results
for res in all_results:
    m = res["method"]
    tag = res["hp_tag"]
    f1 = res["results"]["test_macro_f1"]
    
    if m not in best_per_method or f1 > best_per_method[m]["f1"]:
        best_per_method[m] = {"f1": f1, "tag": tag, "result": res}

for method, info in best_per_method.items():
    res = info["result"]
    hp = res["hyperparams"]
    
    model, params = build_peft_model(
        method=method,
        checkpoint_path=LEAF_JEPA_CHECKPOINT,
        model_name=VIT_MODEL_NAME,
        embed_dim=VIT_EMBED_DIM,
        num_classes=NUM_CLASSES,
        device=device,
        rank=hp.get("rank", 8),
        bottleneck_dim=hp.get("bottleneck_dim", 64),
        num_prompts=hp.get("num_prompts", 50),
        num_layers=VIT_DEPTH,
    )
    
    latency = profile_inference(model, device)
    
    profile_results[method] = {
        "inference_ms": latency,
        "trainable_params": params["trainable"],
        "peak_vram_mb": res["compute"]["peak_vram_mb"],
        "avg_epoch_time_s": res["compute"]["avg_epoch_time_s"],
    }
    
    print(f"  {method:<15} | {latency:6.2f} ms/img | "
          f"VRAM: {res['compute']['peak_vram_mb']:,.0f} MB | "
          f"Params: {params['trainable']:,}")
    
    del model
    torch.cuda.empty_cache()

save_results(profile_results, SET1_DIR / "compute_profile.json")

## Summary Table

In [ ]:
import pandas as pd

rows = []

for method, info in best_per_method.items():
    tag = info["tag"]
    
    # Get all seed results for this method's best HP
    method_seed_results = [
        r for r in all_results
        if r["method"] == method and r["hp_tag"] == tag
    ]
    agg = aggregate_seed_results(method_seed_results)
    prof = profile_results.get(method, {})
    
    rows.append({
        "Method": f"{method} ({tag})",
        "Macro-F1": f"{agg['macro_f1']['mean']:.4f} ± {agg['macro_f1']['std']:.4f}",
        "Accuracy": f"{agg['accuracy']['mean']:.4f} ± {agg['accuracy']['std']:.4f}",
        "Trainable Params": f"{agg['param_count']['trainable']:,}",
        "% Trainable": f"{agg['param_count']['pct_trainable']:.4f}%",
        "VRAM (MB)": f"{prof.get('peak_vram_mb', 'N/A'):,.0f}",
        "Inference (ms)": f"{prof.get('inference_ms', 'N/A'):.2f}",
        "Epoch Time (s)": f"{prof.get('avg_epoch_time_s', 'N/A'):.1f}",
    })

summary_df = pd.DataFrame(rows)

print("\n" + "=" * 100)
print("  SET 1: METHOD COMPARISON SUMMARY (100% labels, best HP per method)")
print("=" * 100)
print(summary_df.to_string(index=False))
print()

summary_df.to_csv(PEFT_DIR / "set1_summary_table.csv", index=False)
print(f"Saved: {PEFT_DIR / 'set1_summary_table.csv'}")

## Pareto Overview Plot

In [ ]:
# Build Pareto data from all configs
pareto_data = []
for res in all_results:
    pareto_data.append({
        "method": res["method"],
        "trainable_params": res["param_count"]["trainable"],
        "macro_f1": res["results"]["test_macro_f1"],
        "label": f"{res['method']} ({res['hp_tag']})",
    })

plot_pareto(
    pareto_data,
    save_path=FIGURES_DIR / "set1_pareto_overview.png",
    title="Set 1: Macro-F1 vs Trainable Parameters (All Configs)",
)

print("\n✅ Set 1 complete.")
print("   Review the summary table above to select top 2 methods for Set 2.")
print("   The Pareto plot is saved to:", FIGURES_DIR / "set1_pareto_overview.png")